# Intro 04 — Counting Techniques

Practice notebook for the **"Counting Techniques"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — The Multiplication Principle

If a process has \(n_1\) options for the first step, \(n_2\) for the second, and so on, the total number of composite outcomes is

$$n_1 \times n_2 \times \cdots \times n_k.$$

The key assumption is that the choices are **independent** — the number of options at step \(i\) does not depend on what was chosen earlier.

**PDF example.** A 3-character password: first character a digit 0–9 (10 choices), the next two lowercase letters a–z (26 each). Total: \(10 \times 26 \times 26 = 6760\).


In [ ]:
import math
from itertools import product

# Password: digit + 2 letters
total = 10 * 26 * 26
print(f"10 x 26 x 26 = {total}  (theory 6760)")

# Enumerate explicitly and confirm — itertools.product is the multiplication principle
digit = [str(d) for d in range(10)]
lower = [chr(c) for c in range(ord('a'), ord('z') + 1)]
all_pws = list(product(digit, lower, lower))
print(f"len(itertools.product) = {len(all_pws)}  -> matches")

# How many of these passwords contain at least one 'z'?
has_z = sum(1 for d, a, b in all_pws if ('z' in (a, b)))
print(f"passwords with at least one 'z': {has_z}")
print(f"sanity: 10*26*26 - 10*25*25 = {10*26*26 - 10*25*25}  (complement: no 'z' = 10*25*25)")

# Plot how total passwords grow as we add more letter slots
k = np.arange(1, 7)
totals = 10 * 26 ** k
fig, ax = plt.subplots()
ax.semilogy(k, totals, 'o-', color='tab:blue')
ax.set_xlabel("number of letter slots (after one digit)")
ax.set_ylabel("total passwords (log scale)")
ax.set_title("Multiplication principle: 1 digit + k letters")
ax.grid(True, which='both', ls=':', alpha=0.6)
plt.show()


**Your turn:** Re-do the count when the first character may be *either* a digit or a letter (36 choices) and the remaining two are letters. How does the total change? Verify with `itertools.product`.


---
# Part 2 — Permutations

A **permutation** is an ordered arrangement. The number of permutations of \(n\) distinct objects taken \(k\) at a time is

$$P(n,k) = n(n-1)\cdots(n-k+1) = \frac{n!}{(n-k)!}.$$

Order matters: gold vs. silver is a different permutation.

**PDF example.** Gold, silver, bronze among 10 runners: \(P(10,3) = 10\cdot 9\cdot 8 = 720\).


In [ ]:
import math
from itertools import permutations

# P(n, k) via math.perm (Python 3.8+)
n, k = 10, 3
print(f"math.perm({n},{k})  = {math.perm(n, k)}   (theory 720)")
print(f"n!/(n-k)!         = {math.factorial(n)//math.factorial(n-k)}")
print(f"product form      = {n*(n-1)*(n-2)}")

# Enumerate all ordered podiums from 10 runners labeled 0..9
runners = list(range(10))
pods = list(permutations(runners, k))
print(f"len(permutations) = {len(pods)}")
print("first 5 podiums (gold,silver,bronze):", pods[:5])

# Permutations grow fast: P(n, n) = n!
print("\n  n    P(n,n)=n!      P(n,5)")
for nn in range(1, 11):
    print(f"  {nn:2d}  {math.factorial(nn):>12d}  {math.perm(nn,5):>10d}")
print("\nNote: P(6,5) = 6! = 720, same as the podium count — choosing all but one.")


**Your turn:** Compare \(P(10,3)\) and \(\binom{10}{3}\). The ratio should be \(k! = 6\). Explain in one sentence why dividing an ordered count by \(k!\) gives the unordered count.


---
# Part 3 — Combinations

A **combination** is a selection where order does **not** matter:

$$\binom{n}{k} = \frac{n!}{k!\,(n-k)!}.$$

Two fundamental identities:

- **Symmetry:** \(\binom{n}{k} = \binom{n}{n-k}\) — choosing what is *in* is equivalent to choosing what is *out*.
- **Pascal's rule:** \(\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k}\).

**PDF example.** 6-card hands from a 52-card deck: \(\binom{52}{6}\).


In [ ]:
import math
from itertools import combinations
from scipy.special import comb

# C(52, 6) three ways
n, k = 52, 6
print(f"math.comb(52,6)    = {math.comb(n, k)}")
print(f"scipy comb(exact)  = {int(comb(n, k, exact=True))}")
print(f"n!/(k!(n-k)!)      = {math.factorial(n)//(math.factorial(k)*math.factorial(n-k))}")

# Verify symmetry C(n,k) == C(n,n-k) for several n,k
print("\nSymmetry C(n,k) == C(n,n-k):")
for nn, kk in [(10,3), (52,6), (20,7), (9,4)]:
    print(f"  C({nn},{kk})={math.comb(nn,kk)}, C({nn},{nn-kk})={math.comb(nn,nn-kk)} -> "
          f"{'OK' if math.comb(nn,kk)==math.comb(nn,nn-kk) else 'FAIL'}")

# Verify Pascal's rule for n=10
print("\nPascal's rule for n=10:")
N = 10
ok = all(math.comb(N, kk) == math.comb(N-1, kk-1) + math.comb(N-1, kk) for kk in range(1, N))
print(f"  holds for all 1<=k<{N}: {ok}")

# Pascal's triangle up to row 7
print("\nPascal's triangle:")
for r in range(8):
    row = [math.comb(r, j) for j in range(r+1)]
    print("  " + " ".join(f"{x:4d}" for x in row))

# Enumerate 6-card hands is too many to list, but 5-card from a 20-card mini-deck is fine
mini_deck = list(range(20))
hands = list(combinations(mini_deck, 5))
print(f"\nMini-deck: C(20,5) = {math.comb(20,5)}, enumerated = {len(hands)}")


**Your turn:** Plot \(\binom{n}{k}\) as a function of \(k\) for \(n=20\). At which \(k\) is the binomial coefficient maximized? Confirm that this \(k^*\) equals \(\lfloor n/2 \rfloor\) or \(\lceil n/2 \rceil\).


---
# Part 4 — Counting and Probability

If all outcomes in a finite sample space are equally likely and event \(A\) contains \(M\) of \(N\) outcomes, then

$$P(A) = \frac{M}{N}.$$

**PDF example.** Draw 6 cards from a 52-card deck; what is \(P(\text{all 6 are hearts})\)? Favorable: \(\binom{13}{6}\); total: \(\binom{52}{6}\), so

$$P(\text{6 hearts}) = \frac{\binom{13}{6}}{\binom{52}{6}} \approx 4.8\times 10^{-5}.$$

This is small precisely because concentrating all cards in one suit is rare.


In [ ]:
import math

# Exact probability all 6 cards are hearts
P_6hearts = math.comb(13, 6) / math.comb(52, 6)
print(f"C(13,6)/C(52,6) = {math.comb(13,6)}/{math.comb(52,6)} = {P_6hearts:.6e}")

# Compare a few 6-card poker-style events
def prob(favorable_comb):
    return favorable_comb / math.comb(52, 6)

events = {
    "all 6 hearts":           math.comb(13, 6),
    "all 6 same suit (any)":  4 * math.comb(13, 6),
    "6 aces?":                 0,                      # only 4 aces exist
    "exactly 4 aces + 2 kings": math.comb(4,4)*math.comb(4,2),
    "exactly 2 aces":          math.comb(4,2)*math.comb(48,4),
}
for name, M in events.items():
    print(f"  {name:28s}: M={M:>12d}, P={prob(M):.6e}")

# Simulate: deal 6-card hands and estimate P(all 6 hearts)
rng = np.random.default_rng(123)
N_SIM = 400_000
deck = np.arange(52)                 # 0..12 -> hearts, 13..25 -> diamonds, etc.
suits = deck // 13
count_all_hearts = 0
for _ in range(N_SIM):
    hand = rng.choice(deck, size=6, replace=False)
    if np.all(suits[hand] == 0):     # 0 = hearts
        count_all_hearts += 1
p_sim = count_all_hearts / N_SIM
print(f"\nSimulation: P(all 6 hearts) ~= {p_sim:.6e}  (theory {P_6hearts:.6e})")
print(f"  -> about {count_all_hearts} hits in {N_SIM:,} deals")


**Your turn:** Modify the simulation to estimate \(P(\text{all 6 same suit})\) — any of the four suits. The exact answer is \(4\binom{13}{6}/\binom{52}{6}\). Does your simulated value land within Monte Carlo error of it?


---
# Part 5 — The Birthday Problem

A classic counting-and-probability result. In a room of \(m\) people (ignoring leap years, so \(N=365\) equally-likely birthdays), the probability that **at least two share a birthday** is best computed via the complement — *all* birthdays distinct:

$$P(\text{all distinct}) = \frac{365\cdot 364\cdots(365-m+1)}{365^m} = \frac{P(365,m)}{365^m},$$

$$P(\text{match}) = 1 - \frac{P(365,m)}{365^m}.$$

Surprisingly, \(P(\text{match})\) exceeds \(\tfrac12\) already at \(m=23\).


In [ ]:
import math

def p_match(m, N=365):
    # P(all distinct) = P(N,m) / N^m, computed in log space for stability
    log_p_distinct = sum(math.log(N - i) for i in range(m)) - m * math.log(N)
    return 1.0 - math.exp(log_p_distinct)

for m in [10, 20, 23, 30, 40, 50, 70]:
    print(f"  m={m:3d}: P(at least one shared birthday) = {p_match(m):.4f}")

print(f"\nm=23 gives P = {p_match(23):.4f}  -> already above 0.5")

# Plot
m_vals = np.arange(1, 80)
p_vals = np.array([p_match(m) for m in m_vals])

fig, ax = plt.subplots()
ax.plot(m_vals, p_vals, color='tab:blue', label='exact $1 - P(365,m)/365^m$')
ax.axhline(0.5, color='tab:red', ls='--', lw=1, label='0.5')
ax.axvline(23, color='tab:green', ls=':', lw=1, label='m = 23')
ax.set_xlabel("number of people m")
ax.set_ylabel("P(at least one shared birthday)")
ax.set_title("Birthday problem")
ax.legend()
plt.show()

# Monte Carlo check at m = 23
rng = np.random.default_rng(7)
N_SIM = 200_000
m = 23
hits = 0
for _ in range(N_SIM):
    bdays = rng.integers(0, 365, size=m)
    if len(set(bdays)) < m:      # at least one duplicate
        hits += 1
p_sim = hits / N_SIM
print(f"\nSimulation at m=23: P ~= {p_sim:.4f}  (theory {p_match(m):.4f})")


**Your turn:** Repeat the exact calculation with \(N=366\) (leap-year model). Does the crossover \(m^*\) change? Try also \(N=12\) (months only) — at what \(m\) does a shared month become more likely than not?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. A license plate has the form LLL-DDDD (3 letters then 4 digits). How many distinct plates are possible? How many if no letter may repeat?

2. Eight runners race. How many ways can gold, silver, and bronze be awarded? How many ways can we simply choose the top-3 finishers (no ordering)? Verify both numbers in Python.

3. How many 5-card poker hands can be dealt from a 52-card deck? Compute the exact number, and give the probability that a uniformly random hand is a **flush** (all 5 cards the same suit).

4. A committee of 4 is chosen from 6 men and 5 women. How many committees have exactly 2 men and 2 women? How many have at least one woman?

5. In a room of 30 people (365 equally-likely birthdays, no leap day), compute the exact probability of at least one shared birthday. Then simulate with 200,000 trials and a fixed seed and report the empirical estimate.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** No-restriction plates: \(26^3 \times 10^4 = 17{,}576{,}000\). With no repeated letters among the 3 letter slots: \(P(26,3)\times 10^4 = 26\cdot 25\cdot 24 \times 10^4 = 15{,}600\times 10^4 = 156{,}000{,}000\).

```python
import math
print(26**3 * 10**4)                      # 17576000
print(math.perm(26,3) * 10**4)            # 156000000
```

**2.** Gold/silver/bronze is ordered: \(P(8,3) = 8\cdot 7\cdot 6 = 336\). Top-3 unordered: \(\binom{8}{3} = 56\). Ratio is \(3! = 6\), exactly the number of orderings of 3 people.

```python
import math
print(math.perm(8,3), math.comb(8,3))     # 336 56
```

**3.** Total 5-card hands: \(\binom{52}{5} = 2{,}598{,}960\). Flush: choose 1 of 4 suits, then 5 of its 13 cards: \(4\binom{13}{5} = 5148\). (This includes straight flushes; a strict non-straight flush subtracts 40.) Probability \(\approx 5148/2{,}598{,}960 \approx 0.001981\).

```python
import math
M = math.comb(4,1) * math.comb(13,5)
N = math.comb(52,5)
print(M, N, M/N)   # 5148 2598960 0.001981...
```

**4.** Exactly 2 men & 2 women: \(\binom{6}{2}\binom{5}{2} = 15\cdot 10 = 150\). At least one woman = total minus all-men: \(\binom{11}{4} - \binom{6}{4} = 330 - 15 = 315\).

```python
import math
print(math.comb(6,2)*math.comb(5,2))                       # 150
print(math.comb(11,4) - math.comb(6,4))                    # 315
print(math.comb(5,1)*math.comb(6,3) + math.comb(5,2)*math.comb(6,2)
      + math.comb(5,3)*math.comb(6,1) + math.comb(5,4)*math.comb(6,0))  # 315 (direct check)
```

**5.** Exact (log-space): \(P = 1 - \exp\big(\sum_{i=0}^{29}\log(365-i) - 30\log 365\big) \approx 0.7063\).

```python
import math, numpy as np
def p_match(m, N=365):
    lp = sum(math.log(N-i) for i in range(m)) - m*math.log(N)
    return 1 - math.exp(lp)
print(p_match(30))   # ~0.7063

rng = np.random.default_rng(2024)
hits = sum(1 for _ in range(200_000) if len(set(rng.integers(0,365,size=30))) < 30)
print(hits/200_000)  # ~0.706
```

</details>
